In [1]:
import Pkg
Pkg.activate("..")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D`


In [2]:
using JLD2
using StaticArrays
using WaveGreen2D: setwave!, wave
using WaveGreen2D.Chebyshev: ChebyshevSeries, gradient, hessian, contains, reduce

In [3]:
setwave!(depth=1.0, wavenumber=1.0)

Wave parameters: h = 1.0 m, ω = 2.73 rad/s, k₀ = 1.0 m, g = 9.81 m/s²

In [4]:
# cs_file = joinpath(@__DIR__, "chebyshev_series.jld2")
cs_file = joinpath("../src/chebyshev_series.jld2")
cs_jld2 = jldopen(cs_file)

const L₁_series = read(cs_jld2, "L₁_series")
const L₂_series = read(cs_jld2, "L₂_series")

close(cs_jld2)

In [5]:
mutable struct ReducedChebyshevSeries
    L₁::ChebyshevSeries{Float64, 2}
    L₂::ChebyshevSeries{Float64, 2}
end

# Reduced series initizalizer
const integrals = ReducedChebyshevSeries(
    ChebyshevSeries(
        Array{Float64, 2}(undef, 1, 1),
        zero(SVector{2, Float64}),
        zero(SVector{2, Float64})
    ),
    ChebyshevSeries(
        Array{Float64, 2}(undef, 1, 1),
        zero(SVector{2, Float64}),
        zero(SVector{2, Float64})
    ),
);

In [6]:
function reduce_integrals(h::Real, ω::Real, g::Real)
    H = Float64(h * ω^2/g)
    H̃ = log(H)

    x = SVector{3, Float64}(0.0, 0.0, H)
    x̃ = SVector{3, Float64}(0.0, 0.0, H̃)

    if contains(L₁_series[1], x)
        integrals.L₁ = clenshaw(L₁_series[1], H)
    elseif contains(L₁_series[2], x̃)
        integrals.L₁ = clenshaw(L₁_series[2], H̃)
    elseif contains(L₁_series[3], x̃)
        integrals.L₁ = clenshaw(L₁_series[3], H̃)
    else
        @warn """Green function and its derivatives might be innacurate in the near field \
                 domain due to the very low value of depth compared to the wavelength"""
        L₁ = clenshaw(L₁_series[1], H)
    end

    if contains(L₂_series[1], x̃)
        integrals.L₂ = clenshaw(L₂_series[1], H̃)
    elseif contains(L₂_series[2], x̃)
        integrals.L₂ = clenshaw(L₂_series[2], H̃)
    elseif contains(L₂_series[3], x̃)
        integrals.L₂ = clenshaw(L₂_series[3], H̃)
    elseif contains(L₂_series[4], x̃)
        integrals.L₂ = clenshaw(L₂_series[4], H̃)
    else
        @warn """Green function and its derivatives might be innacurate in the near field \
                 domain due to the very low value of depth compared to the wavelength"""
        integrals.L₂ = clenshaw(L₂_series[1], H̃)
    end
    
    return nothing
end;

In [7]:
integrals.L₁

2-dimensional Chebyshev series of order (0, 0) for x ∈ [0.0, 0.0]×[0.0, 0.0]

In [8]:
integrals.L₂

2-dimensional Chebyshev series of order (0, 0) for x ∈ [0.0, 0.0]×[0.0, 0.0]

In [11]:
reduce_integrals(wave.h, wave.ω, wave.g)

In [12]:
integrals.L₁

2-dimensional Chebyshev series of order (14, 19) for x ∈ [0.0, 0.5]×[0.0, 1.0]

In [13]:
integrals.L₂

2-dimensional Chebyshev series of order (11, 18) for x ∈ [0.0, 0.5]×[0.0, 2.0]

In [22]:
function Λ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/wave.h

    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/wave.h

    u = SVector{2, Float64}(A, B₁)

    return integrals.L₁(u)
end


function ∇Λ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/wave.h
    dA_dx = sign(R̄)/wave.h
    
    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/wave.h
    dB₁_dz = sign(v̄₁)/wave.h

    u = SVector{2, Float64}(A, B₁)
    ∇u = SVector{2, Float64}(dA_dx, dB₁_dz)

    L, ∇ᵤL = gradient(integrals.L₁, u)

    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = ∇ᵤL .* ∇u
    
    return L, ∇L
end


function HΛ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/wave.h
    dA_dx = sign(R̄)/wave.h
    
    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/wave.h
    dB₁_dz = sign(v̄₁)/wave.h

    u = SVector{2, Float64}(A, B₁)
    ∇u = SVector{2, Float64}(dA_dx, dB₁_dz)
    ∇uᵈ = SMatrix{2, 2, Float64}([dA_dx 0.0; 0.0 dB₁_dz])

    L, ∇ᵤL, HᵤL = hessian(integrals.L₁, u)

    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = ∇ᵤL .* ∇u

    # ∂²L/∂x² = ∂²L/∂u² ⋅ (∂u/∂x)²
    HL = ∇uᵈ * HᵤL * ∇uᵈ
    
    return L, ∇L, HL
end;

In [23]:
P = SA[1.0, -2.0]
Q = SA[5.0, -1.0];

In [27]:
wave.h * wave.ω^2 / wave.g

0.7615941559557649

In [24]:
Λ₁(P, Q)

4.648371286755628e6

In [25]:
∇Λ₁(P, Q)

(4.648371286755628e6, [1.7603288529547896e7, 6.674445148844132e7])

In [26]:
HΛ₁(P, Q)

(4.648371286755628e6, [1.7603288529547896e7, 6.674445148844132e7], [6.1862333823339805e7 2.513489141409425e8; 2.513489141409425e8 2.290328366785746e9])

In [3]:
function reduce(a::Array{T,N}, b::SVector{N,T}; dim::Int=N) where {T,N}
    perm = ntuple(i -> i < dim ? i : (i < N ? i + 1 : dim), Val(N))

    c = permutedims(a, perm)
    d = deleteat(b, dim)
    
    return c, d
end

reduce (generic function with 1 method)

In [4]:
a = rand(2, 2, 2, 2)
b = SVector{4, Float64}(1.0, 2.0, 3.0, 4.0)

a

2×2×2×2 Array{Float64, 4}:
[:, :, 1, 1] =
 0.0291441  0.545509
 0.999512   0.432336

[:, :, 2, 1] =
 0.498637  0.127035
 0.215903  0.902607

[:, :, 1, 2] =
 0.0385957  0.104523
 0.410783   0.211527

[:, :, 2, 2] =
 0.579981  0.0354832
 0.423183  0.585708

In [5]:
c, d = reduce(a, b; dim=2)

c

2×2×2×2 Array{Float64, 4}:
[:, :, 1, 1] =
 0.0291441  0.498637
 0.999512   0.215903

[:, :, 2, 1] =
 0.0385957  0.579981
 0.410783   0.423183

[:, :, 1, 2] =
 0.545509  0.127035
 0.432336  0.902607

[:, :, 2, 2] =
 0.104523  0.0354832
 0.211527  0.585708

In [6]:
a[1, 1, 2, 1] == c[1, 2, 1, 1]

true

In [7]:
d

3-element SVector{3, Float64} with indices SOneTo(3):
 1.0
 3.0
 4.0

In [8]:
@code_warntype reduce(a, b; dim=2)

MethodInstance for Core.kwcall(::@NamedTuple{dim::Int64}, ::typeof(reduce), ::Array{Float64, 4}, ::SVector{4, Float64})
  from kwcall(::NamedTuple, ::typeof(reduce), a::Array{T, N}, b::SVector{N, T}) where {T, N} @ Main In[3]:1
Static Parameters
  T = Float64
  N = 4
Arguments
  _::Core.Const(Core.kwcall)
  @_2::@NamedTuple{dim::Int64}
  @_3::Core.Const(Main.reduce)
  a::Array{Float64, 4}
  b::SVector{4, Float64}
Locals
  dim::Union{}
  @_7::Int64
Body::Tuple{Array{Float64, 4}, SVector{3, Float64}}
1 ──       Core.NewvarNode(:(dim))
│          Core.NewvarNode(:(@_7))
│    %3  = Core.isdefined(@_2, :dim)::Core.Const(true)
└───       goto #6 if not %3
2 ── %5  = Core.getfield(@_2, :dim)::Int64
│    %6  = Main.Int::Core.Const(Int64)
│    %7  = (%5 isa %6)::Core.Const(true)
└───       goto #4 if not %7
3 ──       goto #5
4 ──       Core.Const(:(Main.Int))
│          Core.Const(:(%new(Core.TypeError, Symbol("keyword argument"), :dim, %10, %5)))
└───       Core.Const(:(Core.throw(%11)))
5 ┄─

In [21]:
# """
#     reduce(f::ChebyshevSeries{T,N}, x::T; dim::Int=N) where {T,N} -> ChebyshevSeries{T,N-1}

# Evalautes a `N`-dimensional Chebyshev series at a value of `x` of
# its dimension `dim`, thus reducing the series dimension to `N-1`.
# """
# function reduce(f::ChebyshevSeries{T,N}, x::T; dim::Int=N) where {T,N}
#     perm = ntuple(i -> i < dim ? i : (i < N ? i + 1 : dim), Val(N))

#     x̄ = (2x - f.lb[dim] - f.ub[dim]) / (f.ub[dim] - f.lb[dim])
    
#     coefs = clenshaw(permutedims(f.coefs, perm), x̄)
#     lb = deleteat(f.lb, dim)
#     ub = deleteat(f.ub, dim)
    
#     return ChebyshevSeries(coefs, lb, ub)
# end;

In [5]:
L₁ = L₁_series[1]

3-dimensional Chebyshev series of order (14, 19, 19) for x ∈ [0.0, 0.5]×[0.0, 1.0]×[0.01, 1.64]

In [6]:
H = 0.3
L₁_2D = reduce(L₁, H; dim=3)

B = 0.5
L₁_1D = reduce(L₁_2D, B; dim=2)

A = 0.1
L₁_0D = reduce(L₁_1D, A; dim=1)

L₁_0D - L₁(SVector{3, Float64}(A, B, H))

0.0

In [7]:
B = 0.5
L₁_2D = reduce(L₁, B; dim=2)

H = 0.3
L₁_1D = reduce(L₁_2D, H; dim=2)

A = 0.1
L₁_0D = reduce(L₁_1D, A; dim=1)

L₁_0D - L₁(SVector{3, Float64}(A, B, H))

0.0

In [8]:
A = 0.1
L₁_2D = reduce(L₁, A; dim=1)

B = 0.5
L₁_1D = reduce(L₁_2D, B; dim=1)

H = 0.3
L₁_0D = reduce(L₁_1D, H)

L₁_0D - L₁(SVector{3, Float64}(A, B, H))

2.220446049250313e-16